# llm_service — Przykłady użycia

Ten notebook pokazuje jak korzystać z modułu `llm_service` w codziennej pracy z Azure OpenAI.

## 0. Instalacja zależności

Potrzebne pakiety: `httpx`, `pyyaml`, `pydantic`

In [ ]:
# %pip install httpx pyyaml pydantic

## 1. Konfiguracja

Dwa sposoby — z pliku YAML lub bezpośrednio w kodzie.

In [ ]:
import sys, os

# Dodaj ścieżkę do llm_service jeśli nie jest zainstalowane jako pakiet
sys.path.insert(0, os.path.abspath(".."))  # dostosuj jeśli notebook jest w innym miejscu

from llm_service import LLMConfig, LLMClient

In [ ]:
# --- Sposób A: z pliku YAML ---
# cfg = LLMConfig.from_yaml("config.yaml")

# --- Sposób B: bezpośrednio w kodzie ---
cfg = LLMConfig(
    api_key=os.environ["AZURE_OPENAI_KEY"],
    endpoint="https://your-resource.openai.azure.com",
    model_name="gpt-4.1",
    # temperature=0.3,   # opcjonalne
    # max_tokens=2048,   # opcjonalne
)

print(f"Model: {cfg.model_name}, concurrency: {cfg.concurrency}")

## 2. Proste zapytanie (chat)

In [ ]:
async with LLMClient(cfg) as llm:
    answer = await llm.chat(
        "Czym jest Azure OpenAI Service? Odpowiedz w 2 zdaniach."
    )
    print(answer)

## 3. System prompt

In [ ]:
async with LLMClient(cfg) as llm:
    answer = await llm.chat(
        "Opisz architekturę mikroserwisów",
        system="Jesteś senior architektem. Odpowiadaj zwięźle, po polsku, z bullet pointami."
    )
    print(answer)

## 4. Batch — wiele promptów równolegle

Semaphore (domyślnie 8) kontroluje ile zapytań leci jednocześnie.

In [ ]:
prompts = [
    "Czym jest ETL?",
    "Czym jest RAG w kontekście LLM?",
    "Co to jest vector database?",
    "Wyjaśnij pojęcie fine-tuning.",
    "Czym jest prompt engineering?",
]

async with LLMClient(cfg) as llm:
    results = await llm.batch(
        prompts,
        system="Odpowiadaj w 1-2 zdaniach, po polsku."
    )

for q, a in zip(prompts, results):
    print(f"Q: {q}\nA: {a}\n")

## 5. JSON output (bez schematu)

In [ ]:
async with LLMClient(cfg) as llm:
    data = await llm.chat_json(
        "Podaj 3 największe miasta w Polsce z populacją. Zwróć JSON z kluczem 'cities', każde miasto to obiekt z 'name' i 'population'.",
    )
    print(data)

## 6. Structured Output z Pydantic

Definiujesz model Pydantic → LLM zwraca dane w tej strukturze → dostajesz gotowy obiekt.

In [ ]:
from pydantic import BaseModel, Field


class City(BaseModel):
    name: str = Field(description="Nazwa miasta")
    country: str = Field(description="Kraj")
    population: int = Field(description="Populacja")


class CityList(BaseModel):
    cities: list[City] = Field(description="Lista miast")


async with LLMClient(cfg) as llm:
    result = await llm.structured(
        "Podaj 5 największych miast w Europie.",
        CityList,
        system="Zwracaj dokładne dane."
    )

# result to obiekt CityList — można normalnie iterować
for city in result.cities:
    print(f"{city.name} ({city.country}): {city.population:,}")

## 7. Ekstrakcja danych z dokumentu

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional


class InvoiceData(BaseModel):
    invoice_number: str = Field(description="Numer faktury")
    seller: str = Field(description="Nazwa sprzedawcy")
    buyer: str = Field(description="Nazwa kupującego")
    total_amount: float = Field(description="Kwota brutto")
    currency: str = Field(description="Waluta")
    issue_date: str = Field(description="Data wystawienia (YYYY-MM-DD)")
    items_count: int = Field(description="Liczba pozycji na fakturze")


document_text = """
FAKTURA VAT nr FV/2026/04/0137
Data wystawienia: 2026-04-08

Sprzedawca: TechSoft Sp. z o.o., ul. Marszałkowska 10, Warszawa
Nabywca: DataCorp S.A., ul. Krakowska 55, Kraków

Pozycje:
1. Licencja Enterprise - 12 000,00 PLN netto
2. Wdrożenie systemu  -  8 500,00 PLN netto
3. Szkolenie zespołu  -  3 200,00 PLN netto

Razem netto: 23 700,00 PLN
VAT 23%:      5 451,00 PLN
Razem brutto: 29 151,00 PLN
"""

async with LLMClient(cfg) as llm:
    invoice = await llm.structured(
        f"Wyciągnij dane z tej faktury:\n\n{document_text}",
        InvoiceData,
        system="Ekstrahujesz ustrukturyzowane dane z dokumentów."
    )

print(f"Faktura: {invoice.invoice_number}")
print(f"Sprzedawca: {invoice.seller}")
print(f"Kupujący: {invoice.buyer}")
print(f"Kwota: {invoice.total_amount} {invoice.currency}")
print(f"Data: {invoice.issue_date}")
print(f"Pozycje: {invoice.items_count}")

## 8. Batch structured — ekstrakcja z wielu dokumentów naraz

In [ ]:
documents = [
    "Umowa nr U/2026/001 zawarta dnia 2026-01-15 pomiędzy ABC Sp. z o.o. a XYZ S.A. na kwotę 150 000 PLN netto.",
    "Umowa nr U/2026/002 zawarta dnia 2026-02-20 pomiędzy DEF Sp. z o.o. a GHI S.A. na kwotę 85 000 EUR netto.",
    "Umowa nr U/2026/003 zawarta dnia 2026-03-10 pomiędzy JKL Sp. z o.o. a MNO S.A. na kwotę 220 000 PLN netto.",
]


class ContractData(BaseModel):
    contract_number: str = Field(description="Numer umowy")
    date: str = Field(description="Data zawarcia (YYYY-MM-DD)")
    party_a: str = Field(description="Pierwsza strona umowy")
    party_b: str = Field(description="Druga strona umowy")
    amount: float = Field(description="Kwota netto")
    currency: str = Field(description="Waluta")


prompts = [
    f"Wyciągnij dane z tej umowy:\n\n{doc}" for doc in documents
]

async with LLMClient(cfg) as llm:
    contracts = await llm.batch_structured(
        prompts,
        ContractData,
        system="Ekstrahujesz dane z umów."
    )

for c in contracts:
    print(f"{c.contract_number}: {c.party_a} ↔ {c.party_b} — {c.amount:,.0f} {c.currency}")

## 9. Porównywanie dwóch plików / tekstów

In [ ]:
class ComparisonResult(BaseModel):
    similarities: list[str] = Field(description="Lista podobieństw")
    differences: list[str] = Field(description="Lista różnic")
    recommendation: str = Field(description="Rekomendacja / podsumowanie")


text_a = """Polityka firmy wersja 1.0: Pracownicy mogą pracować zdalnie 2 dni w tygodniu.
Wymagana jest obecność w biurze w poniedziałki i czwartki. Urlop wynosi 26 dni."""

text_b = """Polityka firmy wersja 2.0: Pracownicy mogą pracować zdalnie 3 dni w tygodniu.
Wymagana jest obecność w biurze w poniedziałki. Urlop wynosi 26 dni. Dodano 2 dni na zdrowie psychiczne."""

async with LLMClient(cfg) as llm:
    comparison = await llm.structured(
        f"Porównaj te dwa dokumenty:\n\n--- WERSJA A ---\n{text_a}\n\n--- WERSJA B ---\n{text_b}",
        ComparisonResult,
        system="Porównujesz dokumenty. Bądź precyzyjny."
    )

print("Podobieństwa:")
for s in comparison.similarities:
    print(f"  • {s}")

print("\nRóżnice:")
for d in comparison.differences:
    print(f"  • {d}")

print(f"\nRekomendacja: {comparison.recommendation}")

## 10. Model reasoning (gpt-5.x)

Klient automatycznie wykrywa model reasoning i dostosowuje parametry.

In [ ]:
# Wystarczy zmienić model_name — reszta dzieje się automatycznie
cfg_reasoning = LLMConfig(
    api_key=os.environ["AZURE_OPENAI_KEY"],
    endpoint="https://your-resource.openai.azure.com",
    model_name="gpt-5.4-mini",
    # reasoning_effort="high",  # opcjonalne: low | medium | high
)

from llm_service import detect_capabilities
caps = detect_capabilities(cfg_reasoning.model_name)
print(f"Model: {cfg_reasoning.model_name}")
print(f"  reasoning: {caps.reasoning}")
print(f"  supports_temperature: {caps.supports_temperature}")
print(f"  supports_system_message: {caps.supports_system_message}")

# Użycie identyczne — klient sam dobiera parametry
# async with LLMClient(cfg_reasoning) as llm:
#     answer = await llm.chat("Rozwiąż to zadanie logiczne: ...")

## 11. Override parametrów per-request

Każde wywołanie `chat()` / `structured()` przyjmuje `**overrides` — można nadpisać dowolny parametr bez zmiany configa.

In [ ]:
async with LLMClient(cfg) as llm:
    # Kreatywna odpowiedź
    creative = await llm.chat(
        "Wymyśl nazwę dla nowego produktu AI",
        temperature=0.9,
        max_tokens=100,
    )
    print(f"Kreatywna: {creative}")

    # Deterministyczna odpowiedź
    precise = await llm.chat(
        "Ile to jest 2+2?",
        temperature=0.0,
        max_tokens=10,
    )
    print(f"Precyzyjna: {precise}")

## 12. Pełna historia konwersacji (multi-turn)

Parametr `messages` pozwala przekazać pełną historię czatu.

In [ ]:
conversation = [
    {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku."},
    {"role": "user", "content": "Czym jest Python?"},
    {"role": "assistant", "content": "Python to język programowania wysokiego poziomu."},
    {"role": "user", "content": "A jakie ma główne zastosowania w data science?"},
]

async with LLMClient(cfg) as llm:
    answer = await llm.chat("", messages=conversation)
    print(answer)

---

# Pipeline — Agent Chains

Poniższe przykłady pokazują jak budować łańcuchy agentów z zależnościami.
Pipeline automatycznie:
- Uruchamia **równolegle** kroki bez zależności (fan-out)
- **Czeka** na wyniki zależności przed uruchomieniem kolejnych kroków (fan-in)
- Wstrzykuje wyniki poprzednich kroków do promptów przez placeholdery `{step_name}`
- Zbiera **trace** z czasem wykonania każdego kroku

## 13. Prosty pipeline — analiza + krytyk

Dwa agenty analizują dokument równolegle, trzeci (krytyk) ocenia ich wyniki.

In [ ]:
from llm_service import Pipeline, Step
from pydantic import BaseModel, Field


# --- Modele wyjściowe dla każdego agenta ---

class BusinessAnalysis(BaseModel):
    strengths: list[str] = Field(description="Mocne strony")
    weaknesses: list[str] = Field(description="Słabe strony")
    opportunities: list[str] = Field(description="Szanse")
    risks: list[str] = Field(description="Ryzyka")

class LegalAnalysis(BaseModel):
    compliance_issues: list[str] = Field(description="Problemy z compliance")
    contract_risks: list[str] = Field(description="Ryzyka kontraktowe")
    recommendations: list[str] = Field(description="Rekomendacje prawne")

class CriticVerdict(BaseModel):
    agreements: list[str] = Field(description="Gdzie analitycy się zgadzają")
    contradictions: list[str] = Field(description="Sprzeczności między analizami")
    final_recommendation: str = Field(description="Końcowa rekomendacja")
    confidence: str = Field(description="Poziom pewności: low/medium/high")


# --- Definicja pipeline ---

pipe = Pipeline(
    # Warstwa 1: dwa agenty pracują RÓWNOLEGLE (brak depends_on)
    Step(
        name="biznes",
        system="Jesteś analitykiem biznesowym. Wykonaj analizę SWOT.",
        output=BusinessAnalysis,
    ),
    Step(
        name="prawnik",
        system="Jesteś prawnikiem korporacyjnym. Oceń ryzyka prawne.",
        output=LegalAnalysis,
    ),
    # Warstwa 2: krytyk CZEKA na oba wyniki
    Step(
        name="krytyk",
        system="Jesteś krytycznym recenzentem. Porównaj analizy, znajdź sprzeczności.",
        prompt="Oto analizy do oceny:\n\n--- ANALIZA BIZNESOWA ---\n{biznes}\n\n--- ANALIZA PRAWNA ---\n{prawnik}\n\nOryginalny dokument:\n{input}",
        output=CriticVerdict,
        depends_on=["biznes", "prawnik"],
    ),
)

# --- Uruchomienie ---

proposal = """
Firma XYZ proponuje przejęcie 60% udziałów w startup ABC za 5 mln PLN.
ABC ma roczne przychody 1.2 mln PLN ale jest nierentowna (strata 300k PLN rocznie).
ABC posiada 3 patenty na technologię AI do analizy dokumentów medycznych.
Umowa zawiera klauzulę non-compete na 5 lat dla założycieli.
Due diligence ujawnił niezapłacone zobowiązania podatkowe za 2024 rok (180k PLN).
"""

async with LLMClient(cfg) as llm:
    result = await pipe.run(llm, input=proposal)

# --- Wyniki ---
print("=== ANALIZA BIZNESOWA ===")
biz = result.output("biznes")
print(f"Mocne strony: {biz.strengths}")
print(f"Ryzyka: {biz.risks}")

print("\n=== ANALIZA PRAWNA ===")
law = result.output("prawnik")
print(f"Compliance: {law.compliance_issues}")
print(f"Rekomendacje: {law.recommendations}")

print("\n=== WERDYKT KRYTYKA ===")
critic = result.output("krytyk")
print(f"Sprzeczności: {critic.contradictions}")
print(f"Rekomendacja: {critic.final_recommendation}")
print(f"Pewność: {critic.confidence}")

print(f"\n{result.trace()}")

## 14. Pipeline 3-warstwowy — ekstrakcja → wzbogacenie → walidacja

Klasyczny wzorzec ETL z LLM: wyciągnij dane, wzbogać, zwaliduj.

In [ ]:
from pydantic import BaseModel, Field
from llm_service import Pipeline, Step


class ExtractedEntities(BaseModel):
    companies: list[str] = Field(description="Nazwy firm")
    people: list[str] = Field(description="Nazwiska osób")
    amounts: list[str] = Field(description="Kwoty z walutą")
    dates: list[str] = Field(description="Daty w formacie YYYY-MM-DD")

class Entity(BaseModel):
    name: str = Field(description="Nazwa encji")
    type: str = Field(description="Typ: company/person/amount/date")
    context: str = Field(description="Kontekst w dokumencie")

class EnrichedData(BaseModel):
    entities: list[Entity] = Field(description="Lista encji z dodatkowym kontekstem")
    relationships: list[str] = Field(description="Relacje między encjami")
    summary: str = Field(description="Podsumowanie w 2-3 zdaniach")

class ValidationReport(BaseModel):
    is_consistent: bool = Field(description="Czy dane są spójne")
    issues: list[str] = Field(description="Znalezione problemy")
    quality_score: int = Field(description="Ocena jakości 1-10")
    corrected_summary: str = Field(description="Poprawione podsumowanie")


etl_pipe = Pipeline(
    # Warstwa 1: ekstrakcja
    Step(
        name="extract",
        system="Wyciągasz encje (firmy, osoby, kwoty, daty) z dokumentów. Bądź precyzyjny.",
        output=ExtractedEntities,
    ),
    # Warstwa 2: wzbogacenie (zależy od ekstrakcji)
    Step(
        name="enrich",
        system="Wzbogacasz wyekstrahowane dane o kontekst i relacje.",
        prompt="Na podstawie tych encji:\n{extract}\n\nOraz oryginalnego dokumentu:\n{input}\n\nDodaj kontekst i opisz relacje.",
        output=EnrichedData,
        depends_on=["extract"],
    ),
    # Warstwa 3: walidacja (zależy od obu poprzednich)
    Step(
        name="validate",
        system="Walidujesz dane. Sprawdź spójność, brakujące informacje, błędy.",
        prompt="Zwaliduj te dane:\n\nEkstrakcja:\n{extract}\n\nWzbogacone:\n{enrich}\n\nOryginalny dokument:\n{input}",
        output=ValidationReport,
        depends_on=["extract", "enrich"],
    ),
)

news_article = """
W dniu 2026-03-15 spółka Orlen S.A. podpisała z firmą SolarTech Sp. z o.o.
umowę na budowę farmy fotowoltaicznej o mocy 50 MW w gminie Przykładów.
Wartość kontraktu to 120 mln PLN netto. Ze strony Orlen umowę podpisał
prezes Daniel Obajtek, a ze strony SolarTech — CEO Anna Kowalska.
Realizacja projektu planowana jest na Q3 2027. Projekt jest współfinansowany
z funduszy EU w ramach programu GreenDeal, kwota dofinansowania: 45 mln EUR.
"""

async with LLMClient(cfg) as llm:
    result = await etl_pipe.run(llm, input=news_article)

print("=== EKSTRAKCJA ===")
ext = result.output("extract")
print(f"Firmy: {ext.companies}")
print(f"Osoby: {ext.people}")
print(f"Kwoty: {ext.amounts}")
print(f"Daty: {ext.dates}")

print("\n=== WZBOGACENIE ===")
enr = result.output("enrich")
print(f"Relacje: {enr.relationships}")
print(f"Podsumowanie: {enr.summary}")

print("\n=== WALIDACJA ===")
val = result.output("validate")
print(f"Spójne: {val.is_consistent}")
print(f"Problemy: {val.issues}")
print(f"Jakość: {val.quality_score}/10")

print(f"\n{result.trace()}")

## 15. Pipeline z plain text (bez Pydantic) + transform

Nie każdy krok musi zwracać structured output. Można mieszać text i Pydantic, a `transform` pozwala post-procesować wynik.

In [ ]:
from llm_service import Pipeline, Step
from pydantic import BaseModel, Field


class FinalScore(BaseModel):
    technical_score: int = Field(description="Ocena techniczna 1-10")
    business_score: int = Field(description="Ocena biznesowa 1-10")
    overall: int = Field(description="Ocena ogólna 1-10")
    go_no_go: str = Field(description="GO lub NO-GO")
    justification: str = Field(description="Uzasadnienie decyzji")


review_pipe = Pipeline(
    # 3 recenzentów równolegle — plain text, każdy pisze swoją opinię
    Step(
        name="tech_review",
        system="Jesteś senior developerem. Oceniasz propozycje techniczne. Pisz zwięźle.",
        # brak output= → zwraca plain text
    ),
    Step(
        name="security_review",
        system="Jesteś ekspertem od security. Szukasz podatności i zagrożeń.",
    ),
    Step(
        name="cost_review",
        system="Jesteś analitykiem kosztów IT. Oceniasz TCO i ROI.",
    ),
    # Finalny scorer — structured output
    Step(
        name="decision",
        system="Na podstawie 3 recenzji podejmij decyzję GO/NO-GO.",
        prompt="Recenzje do oceny:\n\n--- TECH ---\n{tech_review}\n\n--- SECURITY ---\n{security_review}\n\n--- KOSZTY ---\n{cost_review}\n\nOryginalny opis:\n{input}",
        output=FinalScore,
        depends_on=["tech_review", "security_review", "cost_review"],
    ),
)

proposal = """
Propozycja: migracja bazy danych z on-prem PostgreSQL 14 do Azure Cosmos DB (NoSQL).
Powód: skalowanie - obecna baza obsługuje 10k req/s, potrzebujemy 100k req/s.
Dane: 2TB, 50 tabel, 12 mikroserwisów korzysta z bazy.
Budżet: 500k PLN na migrację, deadline: 6 miesięcy.
Zespół: 3 backend devs, 1 DBA, brak doświadczenia z Cosmos DB.
"""

async with LLMClient(cfg) as llm:
    result = await review_pipe.run(llm, input=proposal)

# Plain text reviews
print("=== TECH REVIEW ===")
print(result.output("tech_review")[:300], "...\n")

print("=== SECURITY REVIEW ===")
print(result.output("security_review")[:300], "...\n")

# Structured decision
decision = result.output("decision")
print(f"=== DECYZJA: {decision.go_no_go} ===")
print(f"Tech: {decision.technical_score}/10")
print(f"Biznes: {decision.business_score}/10")
print(f"Overall: {decision.overall}/10")
print(f"Uzasadnienie: {decision.justification}")

print(f"\n{result.trace()}")

## 16. Pipeline na wielu dokumentach (batch + pipeline)

Pipeline można uruchamiać na wielu dokumentach — po prostu w pętli lub z `asyncio.gather`.

In [ ]:
import asyncio
from llm_service import Pipeline, Step
from pydantic import BaseModel, Field


class DocumentSummary(BaseModel):
    title: str = Field(description="Tytuł/temat dokumentu")
    key_facts: list[str] = Field(description="Kluczowe fakty")
    sentiment: str = Field(description="positive/neutral/negative")

class QualityCheck(BaseModel):
    is_accurate: bool = Field(description="Czy podsumowanie jest dokładne")
    missing_info: list[str] = Field(description="Brakujące informacje")


# Pipeline reużywalny — definiujesz raz, odpalisz na 100 dokumentach
qa_pipe = Pipeline(
    Step(
        name="summary",
        system="Podsumowujesz dokumenty. Wyciągaj kluczowe fakty.",
        output=DocumentSummary,
    ),
    Step(
        name="qa",
        system="Sprawdzasz jakość podsumowań. Porównujesz z oryginałem.",
        prompt="Podsumowanie:\n{summary}\n\nOryginalny dokument:\n{input}\n\nCzy podsumowanie jest kompletne i dokładne?",
        output=QualityCheck,
        depends_on=["summary"],
    ),
)

documents = [
    "Raport Q1 2026: przychody wzrosły o 15% r/r do 45 mln PLN. EBITDA: 12 mln PLN. Zatrudnienie: +30 osób.",
    "Notatka ze spotkania: zespół zdecydował o migracji do Kubernetes. Deadline: koniec Q2. Potrzebny 1 DevOps.",
    "Reklamacja klienta: system nie działa od 3 dni. Klient grozi wypowiedzeniem umowy. SLA przekroczone o 48h.",
]

async with LLMClient(cfg) as llm:
    # Uruchom pipeline na wszystkich dokumentach RÓWNOLEGLE
    # (każdy pipeline ma swoje wewnętrzne warstwy, a semaphore kontroluje łączną liczbę requestów)
    all_results = await asyncio.gather(*[
        qa_pipe.run(llm, input=doc) for doc in documents
    ])

for i, (doc, res) in enumerate(zip(documents, all_results)):
    summary = res.output("summary")
    qa = res.output("qa")
    print(f"--- Dokument {i+1} ---")
    print(f"  Tytuł: {summary.title}")
    print(f"  Sentiment: {summary.sentiment}")
    print(f"  Fakty: {summary.key_facts}")
    print(f"  QA OK: {qa.is_accurate}, Braki: {qa.missing_info}")
    print()

---

# Token Usage i Koszty

Klient automatycznie zlicza tokeny i szacuje koszty (USD) po każdym requeście.

## 17. Śledzenie tokenów i kosztów w sesji

`llm.usage` zbiera łączne zużycie tokenów i szacunkowy koszt za całą sesję (async with).

In [ ]:
async with LLMClient(cfg) as llm:
    # Kilka zapytań
    await llm.chat("Czym jest Python?")
    await llm.chat("Czym jest Rust?")
    await llm.batch(["Czym jest Go?", "Czym jest Java?", "Czym jest C#?"])

    # Sprawdź zużycie
    print(llm.usage.summary())
    print()
    print(f"Bezpośredni dostęp:")
    print(f"  Requesty:       {llm.usage.request_count}")
    print(f"  Prompt tokens:  {llm.usage.prompt_tokens:,}")
    print(f"  Compl. tokens:  {llm.usage.completion_tokens:,}")
    print(f"  Total tokens:   {llm.usage.total_tokens:,}")
    print(f"  Koszt:          ${llm.usage.cost_usd:.4f} USD")

## 18. Koszty per step w pipeline

`result.trace()` teraz pokazuje tokeny i koszt per krok. Dostępne też jako `result.total_tokens` i `result.total_cost_usd`.

In [ ]:
from llm_service import Pipeline, Step
from pydantic import BaseModel, Field


class Summary(BaseModel):
    title: str = Field(description="Tytul")
    key_points: list[str] = Field(description="Kluczowe punkty")

class Review(BaseModel):
    is_complete: bool = Field(description="Czy kompletne")
    score: int = Field(description="Ocena 1-10")


cost_pipe = Pipeline(
    Step(name="summarize", system="Podsumowujesz dokumenty.", output=Summary),
    Step(
        name="review",
        system="Oceniasz jakość podsumowań.",
        prompt="Podsumowanie:\n{summarize}\n\nOryginalny dokument:\n{input}",
        output=Review,
        depends_on=["summarize"],
    ),
)

async with LLMClient(cfg) as llm:
    result = await cost_pipe.run(llm, input="Firma ABC ogłosiła przychody 50 mln PLN za Q1 2026, wzrost o 20% r/r.")

    # Trace z tokenami i kosztami per step
    print(result.trace())
    print()
    print(f"Łącznie: {result.total_tokens:,} tokenów, ${result.total_cost_usd:.4f} USD")
    print(f"Cała sesja klienta: {llm.usage.summary()}")

## 19. Sprawdzenie cennika modeli

Cennik jest wbudowany w `usage.py`. Można sprawdzić cenę dowolnego modelu.

In [ ]:
from llm_service import get_pricing

models = ["gpt-4.1", "gpt-4.1-mini", "gpt-4.1-nano", "gpt-4o", "gpt-4o-mini", "gpt-5.4-mini", "o3", "o4-mini"]

print(f"{'Model':<20} {'Input $/1M':>12} {'Output $/1M':>12}")
print("-" * 46)
for name in models:
    p = get_pricing(name)
    if p:
        print(f"{name:<20} ${p[0]:>10.2f} ${p[1]:>10.2f}")
    else:
        print(f"{name:<20} {'n/a':>12} {'n/a':>12}")

---

# Vision / Wielomodalność

Modele GPT-4.1, GPT-4o i nowsze obsługują obrazy. Parametr `images` pozwala dołączyć zdjęcia, skany, screenshoty do dowolnego zapytania.

Obsługiwane źródła:
- **Ścieżka do pliku** — `"scan.png"`, `Path("docs/invoice.jpg")`
- **URL** — `"https://example.com/photo.png"`
- **Bajty** — `open("img.png", "rb").read()`

## 20. Opis obrazu (chat + image)

In [ ]:
# Opis obrazu z pliku lokalnego
async with LLMClient(cfg) as llm:
    answer = await llm.chat(
        "Opisz szczegółowo co widzisz na tym obrazie.",
        images=["path/to/image.png"],   # ścieżka do pliku
        # images=["https://example.com/photo.jpg"],  # lub URL
        # image_detail="high",  # opcjonalnie: "auto" | "low" | "high"
    )
    print(answer)

## 21. OCR + ekstrakcja danych ze skanu (structured + image)

Najczęstszy use-case: skan faktury/dokumentu → Pydantic model.

In [ ]:
from pydantic import BaseModel, Field


class ScannedInvoice(BaseModel):
    invoice_number: str = Field(description="Numer faktury")
    seller: str = Field(description="Nazwa sprzedawcy")
    buyer: str = Field(description="Nazwa kupującego")
    total_gross: float = Field(description="Kwota brutto")
    currency: str = Field(description="Waluta")
    issue_date: str = Field(description="Data wystawienia (YYYY-MM-DD)")


# Ekstrakcja strukturalna ze skanu
async with LLMClient(cfg) as llm:
    invoice = await llm.structured(
        "Wyciągnij dane z tej zeskanowanej faktury.",
        ScannedInvoice,
        system="Ekstrahujesz dane z obrazów dokumentów. Bądź precyzyjny.",
        images=["path/to/scanned_invoice.jpg"],
        image_detail="high",  # wysoka rozdzielczość dla OCR
    )

    print(f"Faktura: {invoice.invoice_number}")
    print(f"Sprzedawca: {invoice.seller}")
    print(f"Kwota: {invoice.total_gross} {invoice.currency}")
    print(f"\nUsage: {llm.usage.summary()}")

## 22. Porównanie dwóch obrazów

Można wysłać wiele obrazów w jednym zapytaniu.

In [ ]:
from pydantic import BaseModel, Field


class ImageComparison(BaseModel):
    similarities: list[str] = Field(description="Podobienstwa miedzy obrazami")
    differences: list[str] = Field(description="Roznice miedzy obrazami")
    verdict: str = Field(description="Ktory obraz jest lepszy i dlaczego")


async with LLMClient(cfg) as llm:
    result = await llm.structured(
        "Porównaj te dwa obrazy. Który jest lepszej jakości?",
        ImageComparison,
        images=[
            "path/to/image_v1.png",
            "path/to/image_v2.png",
        ],
    )
    print(f"Podobienstwa: {result.similarities}")
    print(f"Roznice: {result.differences}")
    print(f"Werdykt: {result.verdict}")

## 23. Batch OCR — wiele skanów równolegle

Łączy batch + vision — np. 50 skanów faktur przetworzonych asynchronicznie.

In [ ]:
from pathlib import Path
from pydantic import BaseModel, Field

class ReceiptData(BaseModel):
    store_name: str = Field(description="Nazwa sklepu")
    total: float = Field(description="Kwota do zapłaty")
    currency: str = Field(description="Waluta")
    date: str = Field(description="Data paragonu (YYYY-MM-DD)")


# Lista skanów paragonów
scan_files = list(Path("scans/").glob("*.jpg"))  # dostosuj ścieżkę

async with LLMClient(cfg) as llm:
    # Każdy skan to osobny request z obrazem — semaphore kontroluje równoległość
    tasks = [
        llm.structured(
            "Wyciągnij dane z tego paragonu.",
            ReceiptData,
            system="Ekstrahujesz dane z paragonów.",
            images=[str(scan)],
            image_detail="high",
        )
        for scan in scan_files
    ]
    results = await asyncio.gather(*tasks, return_exceptions=True)

for scan, result in zip(scan_files, results):
    if isinstance(result, Exception):
        print(f"  BŁĄD {scan.name}: {result}")
    else:
        print(f"  {scan.name}: {result.store_name} — {result.total} {result.currency} ({result.date})")

print(f"\nPrzetworzono {len(scan_files)} skanów")
print(llm.usage.summary())